# 리포트 67 — X410 의 12-bit ADC 동적범위가 직접파 제거의 천장이다

> ### 한 일
> **보유 장비 USRP X410 의 공식 사양을 한 곳에서 읽어 세션이 무엇에 묶이는지를 항목마다 수치로 고정했다.**

### 결과
1. 12-bit ADC(아날로그 신호를 숫자로 바꾸는 변환기)의 동적범위는 74.01 dB [^1] 이고, 이 값이 직접파 제거의 천장이다.
2. 세 파형 중 여유가 가장 좁은 것은 `LTE20` 이고 19.2 dB [^2] 다 — 점유대역이 좁아 기준채널 이득이 높다.
3. 세션은 기준 1 + 감시 1 = 2 채널 [^3] 을 같은 클럭에서 쓴다. 사양의 4 RX 는 각도축을 여는 예비다.
4. 채널당 순시대역은 400 MHz [^4] 이고 주파수 범위는 1 MHz [^5] ~ 7.2 GHz [^6] 로 세 밴드를 전부 덮는다.
5. 이 DNR 은 자유공간 시뮬 기하에서 나온 값이다 — 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 그만큼 줄어든다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사양 출처 | ni.com / ettus.com 공식 스펙 한 곳에서 인용 — `src/experiment_x410.py:61` |
| 기하 배치 | `src/experiment_x410.py:100` 한 곳에 있다 |
| 양자화 잔차 | 직접파를 양자화한 뒤 남는 잔차를 `src/experiment_x410.py:83` 의 `adc_quantize()` 가 모델에 넣는다 |
| DNR 의 출처 | 자유공간 시뮬 기하에서 계산한 값이다 — 야외 실측이 이 값을 대체한다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---

## X410 한 대가 기준과 감시를 동시에 든다

세션은 **RX0 = 기준(직접파)** 과 **RX1 = 감시** 두 채널을 **같은 클럭**에서 쓴다[^7] — 사양의 4 RX 는 각도축을 여는 예비다.

사양은 `src/experiment_x410.py:61`, 기하 배치는 `src/experiment_x410.py:100` 한 곳에 있다.

## 사양이 무엇을 제약하나

| 항목 | 값 | 무엇을 제약하나 |
|---|---|---|
| TX / RX 채널 | 4 [^8] / 4 [^9] | 세션은 기준 1 + 감시 1 을 공통 클럭에서 쓴다 |
| 채널당 순시대역 | 400 MHz [^4] | 거리분해능과 점표적 서브밴드 |
| 주파수 범위 | 1 MHz [^5] ~ 7.2 GHz [^6] | 세 밴드 전부 커버 |
| ADC 동적범위 | 74.01 dB [^1] | 직접파 제거의 천장 |
| 감시배열 AoA 빔폭 | 33.8° [^10] | 네 RX 를 전부 감시로 쓸 때 열리는 각도축 |
| 최대대역 바이스태틱 ΔR | 0.749 m [^11] | 표적이 퍼지는 폭 |

원사양 출처는 `src/experiment_x410.py:61 (ni.com / ettus.com 2024 spec)` 한 곳이다.

## 여유가 가장 좁은 파형

![report06_adc_headroom](../outputs/figures/report06_adc_headroom.png)

**그림 1.** 12-bit ADC 는 직접파 대 잡음비 위에 얼마의 여유를 남기는가?
여유가 가장 좁은 파형은 `LTE20` 이고 19.2 dB [^2] 다 — 점유대역이 좁아 기준채널 이득이 높다.

이 DNR 은 자유공간 시뮬 기하에서 나온 값이다[^12]. 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 그만큼 줄어든다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 직접파를 실제로 받아 ECA 잔차를 잰다 | 여유 19.2 dB [^2] 가 야외에 얼마나 남는지가 측정값으로 확정된다 | [편 75 «결정표»](75_decision-matrix.ipynb) 의 사슬 확인 행 |
| 네 RX 를 전부 감시로 두는 배치를 따로 설계한다 | 각도축이 열리고 그 각도축이 검출 이후의 확장축이 된다 | `src/experiment_x410.py:100` 확장 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report06_measurement.json` | `hw.dynamic_range_db` | 74.01 |
| [^2] | `outputs/report06_measurement.json` | `adc.headroom_db_min` | 19.23 |
| [^3] | `outputs/report06_derived.json` | `layers.n_channels` | 2 |
| [^4] | `outputs/report06_measurement.json` | `hw.max_bw_mhz` | 400 |
| [^5] | `outputs/report06_derived.json` | `hw_span.f_lo_mhz` | 1 |
| [^6] | `outputs/report06_derived.json` | `hw_span.f_hi_ghz` | 7.2 |
| [^7] | `outputs/measurement_layers.json` | `validation_three_points.channels` | reference + surveillance = 2 channels on a common clock… |
| [^8] | `outputs/report06_measurement.json` | `hw.n_tx` | 4 |
| [^9] | `outputs/report06_measurement.json` | `hw.n_rx` | 4 |
| [^10] | `outputs/report06_measurement.json` | `hw.aoa_beamwidth_deg` | 33.84 |
| [^11] | `outputs/report06_measurement.json` | `hw.range_res_bistatic_m_at_max_bw` | 0.7495 |
| [^12] | `outputs/report06_measurement.json` | `adc.dnr_source` | outputs/verify_cfar.json:chain.<wf>.dnr_db (simulated g… |